In [0]:
%pip install "unitycatalog-ai[databricks]"


In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

CATALOG = "cs4603"
SCHEMA = "default"
PREFIX = "s27100380"

client = DatabricksFunctionClient()
print("Unity Catalog packages loaded successfully.")

In [0]:
def growth_rate(
    start_value: float,
    rate: float,
    years: int,
) -> float:
    """Project a starting value using compound annual growth.

    Args:
        start_value: Initial numeric value before growth.
        rate: Annual growth rate expressed as a decimal, such as 0.08 for 8%.
        years: Number of complete years over which growth is compounded.

    Returns:
        The projected value after compound growth.
    """
    return start_value * (1.0 + rate) ** years


def percentage_change(
    old_value: float,
    new_value: float,
) -> float:
    """Calculate the percentage change between two values.

    Args:
        old_value: Original baseline value. It must not be zero.
        new_value: Updated value being compared with the baseline.

    Returns:
        Percentage change, where a positive value means an increase and a
        negative value means a decrease.

    Raises:
        ValueError: If old_value is zero.
    """
    if old_value == 0:
        raise ValueError("old_value must not be zero")

    return ((new_value - old_value) / abs(old_value)) * 100.0


def compare_values(
    first_value: float,
    second_value: float,
) -> str:
    """Compare two numeric values and describe their difference.

    Args:
        first_value: First value to compare.
        second_value: Second value to compare.

    Returns:
        A description identifying the larger value and the absolute difference.
    """
    if first_value == second_value:
        return f"{first_value:g} and {second_value:g} are equal"

    larger = max(first_value, second_value)
    smaller = min(first_value, second_value)
    difference = larger - smaller

    return (
        f"{larger:g} is larger than {smaller:g} "
        f"by an absolute difference of {difference:g}"
    )

In [0]:
python_functions = [
    growth_rate,
    percentage_change,
    compare_values,
]

for function in python_functions:
    created = client.create_python_function(
        func=function,
        catalog=CATALOG,
        schema=SCHEMA,
        replace=True,
    )
    print(f"Registered: {CATALOG}.{SCHEMA}.{function.__name__}")

In [0]:
tests = [
    (
        "cs4603.default.growth_rate",
        {
            "start_value": 16.91,
            "rate": 0.08,
            "years": 3,
        },
    ),
    (
        "cs4603.default.percentage_change",
        {
            "old_value": 100.0,
            "new_value": 125.0,
        },
    ),
    (
        "cs4603.default.compare_values",
        {
            "first_value": 16.91,
            "second_value": 18.60,
        },
    ),
]

for function_name, parameters in tests:
    result = client.execute_function(function_name, parameters)
    print(function_name)
    print(result)
    print("-" * 60)

In [0]:
%sql
CREATE OR REPLACE FUNCTION cs4603.default.to_billions(
    amount DOUBLE
)
RETURNS DOUBLE
COMMENT 'Convert an amount expressed in base units into billions.'
RETURN amount / 1000000000.0;

In [0]:
%sql
SELECT cs4603.default.to_billions(2400000000.0)
       AS amount_in_billions;

In [0]:
from uc_tools.register_functions import (
    register_python_functions,
    verify_python_functions,
)

registered_names = register_python_functions()
verification_results = verify_python_functions()

In [0]:
%pip install -U databricks-langchain "unitycatalog-ai[databricks]" unitycatalog-langchain
dbutils.library.restartPython()

In [0]:
from databricks_langchain import UCFunctionToolkit
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

print("All UC tool packages loaded.")

In [0]:
from dotenv import load_dotenv

load_dotenv(".env")


In [0]:
import importlib
import inspect
import agent.graph_uc as graph_uc

importlib.invalidate_caches()
graph_uc = importlib.reload(graph_uc)

print("LOADED FILE:")
print(graph_uc.__file__)

print("\nBUILD GRAPH USES UC PLANNER:")
print(inspect.getsource(graph_uc.build_graph))

print("\nUC PLANNER SOURCE:")
print(inspect.getsource(graph_uc.make_uc_planner))

In [0]:
from agent.graph_uc import build_graph

uc_graph = graph_uc.build_graph()
result = uc_graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "What was Meridian's FY2023 net revenue, and what "
                    "would it be after 3 years of 8% compound growth?"
                ),
            }
        ]
    }
)

print("Plan:")
print(result["plan"])

print("\nStep results:")
for index, step_result in enumerate(result["step_results"], start=1):
    print(f"Step {index}: {step_result}")

print("\nFinal answer:")
print(result["messages"][-1].content)

In [0]:
%sql
GRANT EXECUTE ON FUNCTION cs4603.default.growth_rate
TO `27100380@lums.edu.pk`;

GRANT EXECUTE ON FUNCTION cs4603.default.percentage_change
TO `27100380@lums.edu.pk`;

GRANT EXECUTE ON FUNCTION cs4603.default.compare_values
TO `27100380@lums.edu.pk`;

GRANT EXECUTE ON FUNCTION cs4603.default.to_billions
TO `27100380@lums.edu.pk`;

In [0]:
%sql
SHOW GRANTS ON FUNCTION cs4603.default.growth_rate;

In [0]:
dbutils.library.restartPython()

In [0]:
import os

os.environ["UC_CATALOG"] = "cs4603"
os.environ["UC_SCHEMA"] = "default"
os.environ["DATABRICKS_MODEL"] = (
    "databricks-meta-llama-3-3-70b-instruct"
)
os.environ["VECTOR_SEARCH_INDEX"] = (
    "cs4603.default.s27100380_analyst_index"
)

In [0]:
from databricks import agents

deployment = agents.deploy(
    model_name="cs4603.default.s27100380_document_analyst_uc",
    model_version=2,
    endpoint_name="s27100380-document-analyst-uc",
    workload_size="Small",
    scale_to_zero=True,
    deploy_feedback_model=False,
    environment_vars={
        "DATABRICKS_MODEL": "databricks-meta-llama-3-3-70b-instruct",
        "VECTOR_SEARCH_INDEX": "cs4603.default.s27100380_analyst_index",
    },
)

print(deployment)

In [0]:
import sys
import os
import importlib.util

# Load the build_tables module directly
module_path = "/Workspace/Users/27100380@lums.edu.pk/27100380_cs4603-pa4/genie/build_tables.py"
spec = importlib.util.spec_from_file_location("build_tables", module_path)
build_tables_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(build_tables_module)

tables = build_tables_module.build_tables(spark)
print(tables)

In [0]:
display(
    spark.sql(
        """
        SELECT
            fiscal_year,
            SUM(revenue_yen) / 1e12 AS revenue_trillion_yen,
            SUM(operating_income_yen) / 1e12
                AS operating_income_trillion_yen
        FROM cs4603.default.s27100380_meridian_segment_financials
        GROUP BY fiscal_year
        ORDER BY fiscal_year
        """
    )
)

In [0]:
display(
    spark.sql(
        """
        SELECT
            fiscal_year,
            line_item,
            amount_yen / 1e12 AS amount_trillion_yen,
            source_note
        FROM cs4603.default.s27100380_meridian_income_statement
        ORDER BY fiscal_year, line_item
        """
    )
)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

for endpoint in w.serving_endpoints.list():
    if endpoint.name.startswith("s27100380-"):
        print("Deleting:", endpoint.name)
        w.serving_endpoints.delete(endpoint.name)

In [0]:
display(
    spark.sql(
        """
        DESCRIBE TABLE EXTENDED
        cs4603.default.s27100380_meridian_segment_financials
        """
    )
)

In [0]:
import importlib
import genie.build_tables

importlib.reload(genie.build_tables)

tables = genie.build_tables.build_tables(spark)
print(tables)

In [0]:
display(spark.sql("""
SELECT
    fiscal_year,
    SUM(revenue_yen) / 1e12 AS total_revenue_trillion_yen,
    SUM(operating_income_yen) / 1e9 AS total_operating_income_billion_yen
FROM cs4603.default.s27100380_meridian_segment_financials
GROUP BY fiscal_year
ORDER BY fiscal_year
"""))

In [0]:
# 1. View the income-statement data
display(
    spark.table(
        "cs4603.default.s27100380_meridian_income_statement"
    ).orderBy("fiscal_year", "line_item")
)

In [0]:
# 2. Verify segment-table columns and comments
display(spark.sql("""
DESCRIBE TABLE EXTENDED
cs4603.default.s27100380_meridian_segment_financials
"""))

In [0]:
# 3. Verify income-statement columns and comments
display(spark.sql("""
DESCRIBE TABLE EXTENDED
cs4603.default.s27100380_meridian_income_statement
"""))

In [0]:
# 4. Confirm both fiscal years exist in both tables
for table in [
    "cs4603.default.s27100380_meridian_segment_financials",
    "cs4603.default.s27100380_meridian_income_statement",
]:
    years = {
        row.fiscal_year
        for row in spark.table(table)
        .select("fiscal_year")
        .distinct()
        .collect()
    }
    print(table, years)
    assert years == {2022, 2023}

In [0]:
# 5. Check the UC deployment status
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint = w.serving_endpoints.get(
    "s27100380-document-analyst-uc"
)

print("Ready:", endpoint.state.ready)
print("Config update:", endpoint.state.config_update)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

spaces = w.genie.list_spaces().spaces or []

for space in spaces:
    print(space.title, space.space_id)

SPACE_ID = next(
    space.space_id
    for space in spaces
    if space.title == "s27100380 Meridian Financial Analyst"
)

print("SPACE_ID:", SPACE_ID)

In [0]:
message = w.genie.start_conversation_and_wait(
    space_id=SPACE_ID,
    content="Which segment had the highest revenue in FY2023?",
)

print("Status:", message.status)
print("Conversation ID:", message.conversation_id)
print("Message ID:", message.message_id)

In [0]:
for attachment in message.attachments or []:
    if attachment.text:
        print("\nANSWER:")
        print(attachment.text.content)

    if attachment.query:
        print("\nGENERATED SQL:")
        print(attachment.query.query)

        query_result = w.genie.get_message_attachment_query_result(
            space_id=SPACE_ID,
            conversation_id=message.conversation_id,
            message_id=message.message_id,
            attachment_id=attachment.attachment_id,
        )

        statement = query_result.statement_response
        rows = (
            statement.result.data_array
            if statement and statement.result
            else []
        )

        print("\nROWS:")
        for row in rows:
            print(row)